In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/raw/tokyo.csv')

In [3]:
df["jnc"] = df["Links to Listing"].str.extract(r"jnc_(\d+)")

In [4]:
# Duplicate values =8002 (expected)

phys = ["Floor", "Rooms", "Square Meters"]
d = df.drop_duplicates()                          # exact copies: expect 8,002 removed
n_exact = len(df) - len(d)

bad = d.groupby("jnc")[phys].nunique().max(axis=1)
conflict = bad[bad > 1].index                     # jnc values with physical conflicts
d = d[~d["jnc"].isin(conflict)]
n_conflict = len(df) - n_exact - len(d)

d = d.drop_duplicates(subset="jnc", keep="first")
print(n_exact, n_conflict, len(d))


8002 10 183490


In [5]:
d.to_csv("../data/interim/listings_dedup.csv", index=False, encoding="utf-8-sig")
print(len(d))

183490


In [7]:
df=pd.read_csv('../data/interim/listings_dedup.csv')

In [ ]:
def parse_yen(text):
    if text == "-":
        return None
    if text.endswith("万円"):
        number = text.replace("万円", "")      # "6.5万円" -> "6.5"
        return round(float(number) * 10000)
    if text.endswith("円"):
        number = text.replace("円", "")        # "3000円" -> "3000"
        return int(number)
    return "FAILED"                            # anything unexpected                    # anything unexpected

In [9]:
print(parse_yen("6.5万円"))    # expect 65000
print(parse_yen("10.2万円"))   # expect 102000
print(parse_yen("3000円"))     # expect 3000
print(parse_yen("-"))          # expect None

65000
102000
3000
None


In [10]:
d["Price_yen"] = d["Price"].apply(parse_yen)
print((d["Price_yen"] == "FAILED").sum())

0


In [ ]:
d["Administrative Costs_yen"] = d["Administrative Costs"].apply(parse_yen)
print((d["Administrative Costs_yen"] == "FAILED").sum())

0


In [13]:
d["Deposit_yen"] = d["Deposit"].apply(parse_yen)
print((d["Deposit_yen"] == "FAILED").sum())

0


In [14]:
d["Gratuity_yen"] = d["Gratuity"].apply(parse_yen)
print((d["Gratuity_yen"] == "FAILED").sum())

0


In [17]:
d["Admin_yen"] = d["Administrative Costs_yen"]  # optional rename
d["Administrative Costs_yen"].describe()

count    150179.000000
mean       6101.231497
std        4356.322585
min           1.000000
25%        3000.000000
50%        5000.000000
75%       10000.000000
max      295000.000000
Name: Administrative Costs_yen, dtype: float64

In [19]:
cols = ["Price_yen", "Administrative Costs_yen", "Deposit_yen", "Gratuity_yen"]
for c in cols:
    d[c] = pd.to_numeric(d[c], errors="coerce")

print(d[cols].describe())
print(d[cols].isna().sum())

          Price_yen  Administrative Costs_yen   Deposit_yen  Gratuity_yen
count  1.834900e+05             150179.000000  1.297200e+05  1.129970e+05
mean   9.931687e+04               6101.231497  1.275848e+05  1.164612e+05
std    7.756559e+04               4356.322585  2.375469e+05  9.100306e+04
min    1.060000e+04                  1.000000  1.000000e+02  1.000000e+02
25%    6.300000e+04               3000.000000  6.600000e+04  7.100000e+04
50%    8.250000e+04               5000.000000  8.750000e+04  9.200000e+04
75%    1.130000e+05              10000.000000  1.280000e+05  1.300000e+05
max    4.450000e+06             295000.000000  1.780000e+07  3.300000e+06
Price_yen                       0
Administrative Costs_yen    33311
Deposit_yen                 53770
Gratuity_yen                70493
dtype: int64
